# 00 - Informacion Ministerio (MINEDUC / MINEDEC)

Panel historico de instituciones educativas del Ecuador, construido a partir de los
archivos de Registros Administrativos del Ministerio de Educacion en `data/external/`
(periodos 2009-2010 a 2025-2026 Inicio).

**Objetivo:** identificar por codigo `AMIE`:
- colegios que cambiaron de **nombre**
- colegios que cambiaron de **ubicacion** (canton/parroquia)
- colegios que se **unificaron/fusionaron** con otro (un AMIE desaparece y sus
  estudiantes/registros pasan a otro AMIE existente)
- colegios **inactivos o cerrados** (AMIE que deja de aparecer y nunca vuelve)

**Fuera de alcance por ahora:** `matricula-pension-mayo-2026.xlsx` queda pendiente de
integrar — primero se valida el tratamiento de los registros de MINEDUC/MINEDEC en
este notebook.


In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from difflib import SequenceMatcher

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent
EXT_DIR = PROJECT_ROOT / "data" / "external"

COLOR_PRIMARIO = "#1F4E79"
COLOR_SECUNDARIO = "#8C9BAB"
COLOR_TERCIARIO = "#5D9BD5"

PATRON_ANIO = re.compile(r"(\d{4})-\d{4}")
PATRON_AMIE = re.compile(r"^\d{2}[A-Z]\d{5}$")

archivos = [
    f for f in EXT_DIR.glob("*RegistrosAdministrativos*.xlsx")
    if not f.name.startswith("~$")
]
archivos_ordenados = sorted(
    archivos, key=lambda f: int(PATRON_ANIO.search(f.name).group(1))
)
print(f"Archivos encontrados: {len(archivos_ordenados)}")
for f in archivos_ordenados:
    print(" -", f.name)


Archivos encontrados: 17
 - MINEDUC_RegistrosAdministrativos_2009-2010-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2010-2011-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2011-2012-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2012-2013-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2013-2014-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2014-2015-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2015-2016-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2016-2017-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2017-2018-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2018-2019-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2019-2020-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2020-2021-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2021-2022-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2022-2023-Fin.xlsx
 - MINEDUC_RegistrosAdministrativos_2023-2024-Fin.xlsx
 - MINEDEC_RegistrosAdministrativos_2024-2025-Fin.xlsx
 - MINEDEC_RegistrosAdministrativos_2025-2026-Inicio.xlsx


## 1. Carga: encontrar la fila de encabezado en cada archivo

Cada archivo tiene unas filas de titulo/metadata antes de la tabla real. Buscamos la fila que contiene literalmente `AMIE`.

In [2]:
def encontrar_fila_encabezado(path, max_filas=30):
    muestra = pd.read_excel(path, header=None, nrows=max_filas, engine="calamine")
    for i, fila in muestra.iterrows():
        valores = [str(v).strip() for v in fila.tolist() if v is not None]
        if "AMIE" in valores:
            return i
    raise ValueError(f"No se encontro fila de encabezado con 'AMIE' en {path.name}")


# Se leen 30 columnas por posicion (no por nombre): las primeras 17 son
# identidad/ubicacion, y se agrega la posicion 29 (Total_Estudiantes = matricula
# total), verificada como estable en los 17 archivos pese a que el resto del
# esquema (desglose por grado) varia bastante entre anios. No hay columna
# "Circuito" en ninguno de los archivos — se usa parroquia/canton como zona.
COLUMNAS_IDENTIDAD = [
    "periodo_raw", "AMIE", "nombre_institucion", "zona",
    "provincia", "cod_provincia", "canton", "cod_canton",
    "parroquia", "cod_parroquia", "_col10", "_col11", "_col12",
    "sostenimiento", "area", "regimen_escolar", "jurisdiccion",
    "_col17", "_col18", "_col19", "_col20", "_col21", "_col22", "_col23",
    "_col24", "_col25", "_col26", "_col27", "_col28",
    "total_estudiantes",
]


def cargar_identidad_colegio(path, fila_encabezado):
    df = pd.read_excel(
        path,
        header=None,
        skiprows=fila_encabezado + 1,
        usecols=list(range(30)),
        engine="calamine",
    )
    df = df.iloc[:, :30]
    df.columns = COLUMNAS_IDENTIDAD
    df = df.drop(columns=[c for c in df.columns if c.startswith("_col")])
    for col in ["cod_provincia", "cod_canton", "cod_parroquia"]:
        df[col] = df[col].astype(str).str.strip()
    for col in ["provincia", "canton", "parroquia", "sostenimiento", "area", "regimen_escolar", "jurisdiccion"]:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": None, "None": None})
    df["AMIE"] = df["AMIE"].astype(str).str.strip()
    df["total_estudiantes"] = pd.to_numeric(df["total_estudiantes"], errors="coerce")
    return df


## 2. Construir el panel historico (2009-2010 a 2025-2026)

In [3]:
registros = []
for path in archivos_ordenados:
    anio_inicio = int(PATRON_ANIO.search(path.name).group(1))
    etiqueta_periodo = PATRON_ANIO.search(path.name).group(0)
    fila_enc = encontrar_fila_encabezado(path)
    df = cargar_identidad_colegio(path, fila_enc)

    validos = df["AMIE"].str.match(PATRON_AMIE, na=False)
    pct_validos = validos.mean() * 100
    if pct_validos < 95:
        print(f"ADVERTENCIA: {path.name} solo {pct_validos:.1f}% de AMIE con formato valido — revisar alineacion de columnas")

    df["anio_inicio"] = anio_inicio
    df["periodo"] = etiqueta_periodo
    df["archivo_origen"] = path.name
    registros.append(df)
    print(f"{path.name}: {len(df):,} filas (header fila {fila_enc}, anio {anio_inicio}, {pct_validos:.1f}% AMIE validos)")

panel = pd.concat(registros, ignore_index=True)
panel = panel.sort_values(["anio_inicio", "AMIE"]).reset_index(drop=True)
print()
print(f"Panel total: {len(panel):,} filas, {panel['AMIE'].nunique():,} AMIE distintos, {panel['anio_inicio'].nunique()} periodos")
panel.head()


MINEDUC_RegistrosAdministrativos_2009-2010-Fin.xlsx: 28,955 filas (header fila 16, anio 2009, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2010-2011-Fin.xlsx: 29,026 filas (header fila 16, anio 2010, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2011-2012-Fin.xlsx: 28,717 filas (header fila 16, anio 2011, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2012-2013-Fin.xlsx: 27,031 filas (header fila 16, anio 2012, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2013-2014-Fin.xlsx: 25,226 filas (header fila 16, anio 2013, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2014-2015-Fin.xlsx: 19,784 filas (header fila 16, anio 2014, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2015-2016-Fin.xlsx: 18,582 filas (header fila 16, anio 2015, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2016-2017-Fin.xlsx: 17,097 filas (header fila 16, anio 2016, 100.0% AMIE validos)
MINEDUC_RegistrosAdministrativos_2017-2018-Fin.xlsx: 16,714 filas (header fila 1

,periodo_raw,AMIE,nombre_institucion,zona,provincia,cod_provincia,canton,cod_canton,parroquia,cod_parroquia,sostenimiento,area,regimen_escolar,jurisdiccion,total_estudiantes,anio_inicio,periodo,archivo_origen
0,2009-2010 Fin,01B00019,CENTRO EDUCATIVO ROUSSEAU,ZONA 6,AZUAY,1,CUENCA,101,BAÑOS,10151,Particular,Rural,Sierra,Intercultural Bilingüe,47,2009,2009-2010,MINEDUC_RegistrosAdministrativos_2009-2010-Fin...
1,2009-2010 Fin,01B00020,COLEGIO INTERCULTURAL BILINGUE DE NARANCAY,ZONA 6,AZUAY,1,CUENCA,101,BAÑOS,10151,Fiscal,Rural,Sierra,Intercultural Bilingüe,264,2009,2009-2010,MINEDUC_RegistrosAdministrativos_2009-2010-Fin...
2,2009-2010 Fin,01B00021,SEIS DE JUNIO,ZONA 6,AZUAY,1,CUENCA,101,CHAUCHA,10153,Fiscal,Rural,Costa,Intercultural Bilingüe,24,2009,2009-2010,MINEDUC_RegistrosAdministrativos_2009-2010-Fin...
3,2009-2010 Fin,01B00022,UNIDAD EDUCATIVA INTERCULTURAL BILINGUE LA PAZ...,ZONA 6,AZUAY,1,CUENCA,101,GIL RAMIREZ DAVALOS,10106,Fiscal,Urbana,Sierra,Intercultural Bilingüe,273,2009,2009-2010,MINEDUC_RegistrosAdministrativos_2009-2010-Fin...
4,2009-2010 Fin,01B00023,LA PAZ,ZONA 6,AZUAY,1,CUENCA,101,MOLLETURO,10157,Fiscal,Rural,Sierra,Intercultural Bilingüe,42,2009,2009-2010,MINEDUC_RegistrosAdministrativos_2009-2010-Fin...


## 3. Normalizar nombres de instituciones

El mismo AMIE puede alternar entre "X" y "UNIDAD EDUCATIVA X" (u otro prefijo
institucional) de un anio a otro sin que sea un cambio real de nombre. Se
normaliza quitando acentos, mayusculas, espacios repetidos y prefijos
institucionales comunes antes de comparar.


In [4]:
PREFIJOS_INSTITUCIONALES = [
    "UNIDAD EDUCATIVA DEL MILENIO",
    "UNIDAD EDUCATIVA INTERCULTURAL BILINGUE",
    "UNIDAD EDUCATIVA BILINGUE",
    "UNIDAD EDUCATIVA FISCOMISIONAL",
    "UNIDAD EDUCATIVA PARTICULAR",
    "UNIDAD EDUCATIVA FISCAL",
    "UNIDAD EDUCATIVA MUNICIPAL",
    "UNIDAD EDUCATIVA",
    "ESCUELA DE EDUCACION BASICA",
    "ESCUELA FISCAL MIXTA",
    "ESCUELA FISCAL",
    "ESCUELA PARTICULAR",
    "ESCUELA MUNICIPAL",
    "ESCUELA",
    "COLEGIO NACIONAL",
    "COLEGIO FISCAL",
    "COLEGIO PARTICULAR",
    "COLEGIO MUNICIPAL",
    "COLEGIO",
    "CENTRO DE EDUCACION BASICA",
    "CENTRO DE EDUCACION INICIAL",
    "CENTRO EDUCATIVO",
    "JARDIN DE INFANTES",
    "INSTITUTO TECNOLOGICO SUPERIOR",
    "INSTITUTO TECNOLOGICO",
    "INSTITUTO SUPERIOR",
    "INSTITUTO",
    "LICEO",
    "CECIB DE EDUCACION BASICA",
    "CECIB DE BACHILLERATO",
    "CECIB",
    "ESCUELA EGB",
    "EXTENSION",
]
PATRON_PREFIJO = re.compile(r"^(" + "|".join(PREFIJOS_INSTITUCIONALES) + r")\s+")

# Valores que el Ministerio usa como placeholder cuando no se registro el
# nombre real de la institucion — no son nombres, se tratan como dato faltante
# para no compararlos por texto contra nombres reales (ver seccion 7: sin
# esto, "SIN NOMBRE" generaba candidatos de fusion falsos como "SAN JOSE"
# solo por parecido superficial de caracteres).
VALORES_PLACEHOLDER_NOMBRE = {
    "SIN NOMBRE",
    "SIN NOMBRE REGISTRADO",
    "S/N",
    "SN",
    "NO REGISTRA",
    "NO APLICA",
}


def normalizar_nombre(nombre):
    if not isinstance(nombre, str) or not nombre.strip():
        return None
    texto = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode("ascii")
    texto = texto.upper().strip()
    texto = re.sub(r"\s+", " ", texto)
    anterior = None
    while anterior != texto:
        anterior = texto
        texto = PATRON_PREFIJO.sub("", texto).strip()
    if texto in VALORES_PLACEHOLDER_NOMBRE:
        return None
    return texto


panel["nombre_normalizado"] = panel["nombre_institucion"].map(normalizar_nombre)

n_placeholder = panel["nombre_institucion"].astype(str).str.strip().str.upper().isin(VALORES_PLACEHOLDER_NOMBRE).sum()
print(f"Filas con nombre placeholder (tratadas como dato faltante): {n_placeholder:,}")

ejemplos = panel[panel["nombre_institucion"].astype(str) != panel["nombre_normalizado"].astype(str)]
ejemplos[["nombre_institucion", "nombre_normalizado"]].drop_duplicates().head(10)


Filas con nombre placeholder (tratadas como dato faltante): 510


,nombre_institucion,nombre_normalizado
0,CENTRO EDUCATIVO ROUSSEAU,ROUSSEAU
1,COLEGIO INTERCULTURAL BILINGUE DE NARANCAY,INTERCULTURAL BILINGUE DE NARANCAY
3,UNIDAD EDUCATIVA INTERCULTURAL BILINGUE LA PAZ...,LA PAZ EXTENSION AZUAY
5,ESCUELA MONSEÑOR LEONIDAS PROAÑO II,MONSENOR LEONIDAS PROANO II
7,SIN NOMBRE,None
10,XAVIER MUÑOZ CHAVEZ,XAVIER MUNOZ CHAVEZ
14,JUAN PEÑAFIEL,JUAN PENAFIEL
21,RUMIÑAHUI,RUMINAHUI
25,MONSEÑOR LEONIDAS PROAÑO 1,MONSENOR LEONIDAS PROANO 1
27,CECIB CESAR A MOROCHO,CESAR A MOROCHO


## 4. Cobertura por anio (numero de instituciones registradas)

In [5]:
cobertura_anual = (
    panel.groupby("anio_inicio")
    .agg(n_instituciones=("AMIE", "nunique"), n_filas=("AMIE", "count"))
    .reset_index()
)

fig_cobertura = go.Figure(
    go.Scatter(
        x=cobertura_anual["anio_inicio"],
        y=cobertura_anual["n_instituciones"],
        mode="lines+markers",
        line=dict(color=COLOR_PRIMARIO, width=2),
        marker=dict(size=8, color=COLOR_PRIMARIO, line=dict(width=2, color="white")),
    )
)
fig_cobertura.update_layout(
    template="plotly_white",
    title="Instituciones educativas registradas por periodo",
    xaxis_title="Anio de inicio del periodo",
    yaxis_title="Instituciones (AMIE unicos)",
    showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10),
)
fig_cobertura.show()
cobertura_anual


,anio_inicio,n_instituciones,n_filas
0,2009,28955,28955
1,2010,29026,29026
2,2011,28717,28717
3,2012,27031,27031
4,2013,25226,25226
5,2014,19784,19784
6,2015,18582,18582
7,2016,17097,17097
8,2017,16714,16714
9,2018,16600,16600


## 5. Historia por AMIE: cambios de nombre, ubicacion e inactividad

Para cada `AMIE` se compara su primera y ultima aparicion, y todos los valores
distintos de nombre normalizado / canton / parroquia que tuvo a lo largo del
tiempo.


In [6]:
anio_maximo = panel["anio_inicio"].max()

historia = (
    panel.sort_values(["AMIE", "anio_inicio"])
    .groupby("AMIE")
    .agg(
        primer_anio=("anio_inicio", "min"),
        ultimo_anio=("anio_inicio", "max"),
        primer_periodo=("periodo", "first"),
        ultimo_periodo=("periodo", "last"),
        nombres=("nombre_normalizado", lambda s: sorted(set(s.dropna()))),
        cantones=("canton", lambda s: sorted(set(s.dropna()))),
        parroquias=("parroquia", lambda s: sorted(set(s.dropna()))),
        ultimo_nombre=("nombre_normalizado", "last"),
        ultimo_canton=("canton", "last"),
        ultima_parroquia=("parroquia", "last"),
        ultima_provincia=("provincia", "last"),
    )
    .reset_index()
)

historia["cambio_nombre"] = historia["nombres"].map(len) > 1
historia["cambio_ubicacion"] = (historia["cantones"].map(len) > 1) | (historia["parroquias"].map(len) > 1)
historia["posible_inactivo"] = historia["ultimo_anio"] < anio_maximo

print(f"AMIE totales: {len(historia):,}")
print(f"Con cambio de nombre detectado: {historia['cambio_nombre'].sum():,}")
print(f"Con cambio de ubicacion detectado: {historia['cambio_ubicacion'].sum():,}")
print(f"Posibles inactivos/cerrados (ultimo periodo visto anterior a {historia['ultimo_periodo'].max()}): {historia['posible_inactivo'].sum():,}")


AMIE totales: 33,206
Con cambio de nombre detectado: 9,958
Con cambio de ubicacion detectado: 4,514
Posibles inactivos/cerrados (ultimo periodo visto anterior a 2025-2026): 16,931


## 6. Transiciones detalladas ano a ano (para revision)

Para los AMIE marcados con cambio de nombre o ubicacion, se listan las transiciones puntuales entre anios consecutivos donde ocurrio el cambio.

In [7]:
amies_con_cambio = historia.loc[historia["cambio_nombre"] | historia["cambio_ubicacion"], "AMIE"]

panel_cambios = panel[panel["AMIE"].isin(amies_con_cambio)].sort_values(["AMIE", "anio_inicio"]).copy()

transiciones = []
for amie, grupo in panel_cambios.groupby("AMIE"):
    grupo = grupo.reset_index(drop=True)
    for i in range(1, len(grupo)):
        anterior = grupo.loc[i - 1]
        actual = grupo.loc[i]
        # Si el nombre no se registro (placeholder -> None) en alguno de los dos
        # anios, no se cuenta como cambio de nombre real — solo como dato faltante.
        cambio_nombre = (
            anterior["nombre_normalizado"] is not None
            and actual["nombre_normalizado"] is not None
            and anterior["nombre_normalizado"] != actual["nombre_normalizado"]
        )
        cambio_ubicacion = (anterior["canton"] != actual["canton"]) or (anterior["parroquia"] != actual["parroquia"])
        if cambio_nombre or cambio_ubicacion:
            transiciones.append({
                "AMIE": amie,
                "anio_anterior": anterior["anio_inicio"],
                "anio_actual": actual["anio_inicio"],
                "nombre_anterior": anterior["nombre_normalizado"],
                "nombre_actual": actual["nombre_normalizado"],
                "canton_anterior": anterior["canton"],
                "canton_actual": actual["canton"],
                "parroquia_anterior": anterior["parroquia"],
                "parroquia_actual": actual["parroquia"],
                "cambio_nombre": cambio_nombre,
                "cambio_ubicacion": cambio_ubicacion,
            })

transiciones_df = pd.DataFrame(transiciones)
print(f"Transiciones detectadas: {len(transiciones_df):,}")
transiciones_df.head(20)


Transiciones detectadas: 20,702


,AMIE,anio_anterior,anio_actual,nombre_anterior,nombre_actual,canton_anterior,canton_actual,parroquia_anterior,parroquia_actual,cambio_nombre,cambio_ubicacion
0,01B00014,2013,2014,BILINGUE INTEGRAL,PARTICULAR INTEGRAL,CUENCA,CUENCA,TOTORACOCHA,TOTORACOCHA,True,False
1,01B00020,2010,2011,INTERCULTURAL BILINGUE DE NARANCAY,INTERCULTURAL BILINGUE DE NARANCAY,CUENCA,CUENCA,BAÑOS,TURI,False,True
2,01B00020,2012,2013,INTERCULTURAL BILINGUE DE NARANCAY,INTERCULTURAL BILINGUE DE NARANCAY,CUENCA,CUENCA,TURI,BAÑOS,False,True
3,01B00020,2018,2019,INTERCULTURAL BILINGUE DE NARANCAY,NARANCAY,CUENCA,CUENCA,BAÑOS,BAÑOS,True,False
4,01B00021,2022,2023,SEIS DE JUNIO,EDUCACION BASICA SEIS DE JUNIO,CUENCA,CUENCA,CHAUCHA,CHAUCHA,True,False
5,01B00022,2012,2013,LA PAZ EXTENSION AZUAY,COMUNITARIA INTERCULTURAL BILINGUE LA PAZ,CUENCA,CUENCA,GIL RAMIREZ DAVALOS,GIL RAMIREZ DAVALOS,True,False
6,01B00030,2013,2014,BUENA ESPERANZA,PARTICULAR BUENA ESPERANZA,CUENCA,CUENCA,RICAURTE,RICAURTE,True,False
7,01B00030,2018,2019,PARTICULAR BUENA ESPERANZA,BUENA ESPERANZA,CUENCA,CUENCA,RICAURTE,RICAURTE,True,False
8,01B00032,2010,2011,XAVIER MUNOZ CHAVEZ,XAVIER MUNOZ CHAVEZ,CUENCA,CUENCA,TARQUI,TURI,False,True
9,01B00032,2023,2024,XAVIER MUNOZ CHAVEZ,XAVIER MUNOZ CHAVEZ,CUENCA,CUENCA,TURI,TARQUI,False,True


## 7. Candidatos a fusion / unificacion

Cuando un AMIE deja de aparecer (posible cierre), se buscan AMIE que **si**
siguen activos en el periodo siguiente, en el **mismo canton**, que cumplan
**todos** estos criterios:

1. Nombre normalizado **practicamente identico** (score de similitud >= 0.97).
   No basta con compartir una palabra o un nombre de pila (ej. "MANUEL ANDRADE
   URETA" vs "MANUEL RIVADENEIRA" da 0.684 y se descarta).
2. El nombre **no** es simplemente el nombre del canton/parroquia/provincia
   (ej. un colegio llamado "PUJILI" en el canton Pujili -- coincidencia de
   toponimo, no evidencia de fusion).
3. La matricula del candidato **crecio** del periodo anterior al periodo
   siguiente (si ya existia antes) -- una fusion real implica que el
   candidato absorbio estudiantes. Si el candidato es un codigo nuevo (no
   existia antes), este criterio no aplica.
4. Solo hay **un unico** candidato que cumple todo lo anterior. Si hay mas de
   uno (ej. varios colegios "VICENTE ROCAFUERTE" en el mismo canton), no
   podemos saber a cual se fusiono realmente -- se trata como cierre, no
   como fusion confirmada.


In [8]:
def similitud(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def normalizar_texto_simple(texto):
    """Solo mayusculas/sin tildes, sin quitar prefijos -- para comparar contra
    nombres de canton/parroquia/provincia (no son nombres de institucion)."""
    if not isinstance(texto, str) or not texto.strip():
        return None
    t = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", t.upper().strip())


# Umbral estricto: solo nombres practicamente identicos tras normalizar cuentan
# como fusion. "MANUEL ANDRADE URETA" vs "MANUEL RIVADENEIRA" (comparten solo
# el nombre de pila) da 0.684 y queda fuera -- no es evidencia de fusion.
UMBRAL_SIMILITUD_FUSION = 0.97

panel_por_anio_canton = {
    (anio, canton): sub for (anio, canton), sub in panel.groupby(["anio_inicio", "canton"])
}
matricula_por_amie_anio = panel.groupby(["AMIE", "anio_inicio"])["total_estudiantes"].first().to_dict()

posibles_cierres = historia[historia["posible_inactivo"]].copy()

candidatos = []
n_descartados_por_nombre_generico = 0
n_descartados_por_no_crecer = 0
for _, fila in posibles_cierres.iterrows():
    anio_siguiente = fila["ultimo_anio"] + 1
    mismo_canton = panel_por_anio_canton.get((anio_siguiente, fila["ultimo_canton"]))
    if mismo_canton is None:
        continue
    mismo_canton = mismo_canton[mismo_canton["AMIE"] != fila["AMIE"]]

    # Nombres "genericos" para este cierre: el propio canton, su provincia y su
    # ultima parroquia -- un colegio llamado igual que su canton/parroquia (ej.
    # "PUJILI" en el canton Pujili) no es evidencia de fusion, es coincidencia
    # de toponimo (ver caso 05H00463/05H00472).
    nombres_genericos_base = {
        normalizar_texto_simple(fila["ultimo_canton"]),
        normalizar_texto_simple(fila["ultima_parroquia"]),
        normalizar_texto_simple(fila["ultima_provincia"]),
    }

    for _, cand in mismo_canton.iterrows():
        score = similitud(fila["ultimo_nombre"], cand["nombre_normalizado"])
        if score < UMBRAL_SIMILITUD_FUSION:
            continue
        nombres_genericos = nombres_genericos_base | {normalizar_texto_simple(cand["parroquia"])}
        if cand["nombre_normalizado"] in nombres_genericos or fila["ultimo_nombre"] in nombres_genericos:
            n_descartados_por_nombre_generico += 1
            continue

        # Si es fusion real, la matricula del candidato debio CRECER del
        # periodo anterior (cuando el colegio que cierra todavia existia) al
        # periodo siguiente (cuando ya lo absorbio). Si el candidato es un
        # codigo nuevo (no existia el periodo anterior), no hay matricula
        # previa que comparar -- no se descarta por este criterio, un codigo
        # nuevo que aparece justo cuando el otro desaparece ya es evidencia
        # fuerte por si sola (ej. caso Rousseau: 01H01665 es nuevo en 2010).
        matricula_cand_antes = matricula_por_amie_anio.get((cand["AMIE"], fila["ultimo_anio"]))
        matricula_cand_despues = cand["total_estudiantes"]
        if pd.isna(matricula_cand_antes):
            crecio = True
        else:
            crecio = pd.notna(matricula_cand_despues) and matricula_cand_despues > matricula_cand_antes
        if not crecio:
            n_descartados_por_no_crecer += 1
            continue

        candidatos.append({
            "amie_desaparecido": fila["AMIE"],
            "nombre_desaparecido": fila["ultimo_nombre"],
            "ultimo_periodo_visto": fila["ultimo_periodo"],
            "canton": fila["ultimo_canton"],
            "amie_candidato": cand["AMIE"],
            "nombre_candidato": cand["nombre_normalizado"],
            "periodo_candidato": cand["periodo"],
            "score_similitud": round(score, 3),
            "matricula_candidato_antes": matricula_cand_antes,
            "matricula_candidato_despues": matricula_cand_despues,
        })

candidatos_df = pd.DataFrame(candidatos)
candidatos_df = candidatos_df.sort_values(
    ["amie_desaparecido", "score_similitud"], ascending=[True, False]
)

print(f"Umbral de similitud usado: >= {UMBRAL_SIMILITUD_FUSION}")
print(f"Descartados por ser nombre generico (igual a canton/parroquia/provincia): {n_descartados_por_nombre_generico:,}")
print(f"Descartados porque la matricula del candidato no crecio: {n_descartados_por_no_crecer:,}")
n_amie_con_candidato = candidatos_df["amie_desaparecido"].nunique() if len(candidatos_df) else 0
print(f"AMIE con al menos un candidato que pasa todos los filtros: {n_amie_con_candidato:,} de {len(posibles_cierres):,} posibles cierres")
candidatos_df.head(20)


Umbral de similitud usado: >= 0.97
Descartados por ser nombre generico (igual a canton/parroquia/provincia): 107
Descartados porque la matricula del candidato no crecio: 892
AMIE con al menos un candidato que pasa todos los filtros: 871 de 16,931 posibles cierres


,amie_desaparecido,nombre_desaparecido,ultimo_periodo_visto,canton,amie_candidato,nombre_candidato,periodo_candidato,score_similitud,matricula_candidato_antes,matricula_candidato_despues
0,01B00019,ROUSSEAU,2009-2010,CUENCA,01H01665,ROUSSEAU,2010-2011,1.00,NaN,46
1,01B00064,FROILAN ABAD AMBROSI,2009-2010,CUENCA,01H01653,FROILAN ABAD AMBROSI,2010-2011,1.00,NaN,13
2,01H00006,FRANCISCA DAVILA DE MUNOZ,2013-2014,CUENCA,01H00046,FRANCISCA DAVILA DE MUNOZ,2014-2015,1.00,583.0,672
3,01H00025,5 DE NOVIEMBRE,2009-2010,CUENCA,01H01679,5 DE NOVIEMBRE,2010-2011,1.00,NaN,25
4,01H00056,RAFAEL CAMPOVERDE GALAN,2010-2011,CUENCA,01H01810,RAFAEL CAMPOVERDE GALAN,2011-2012,1.00,NaN,136
5,01H00079,JULIO ABAD CHICA,2012-2013,CUENCA,01H00067,JULIO ABAD CHICA,2013-2014,1.00,484.0,856
6,01H00106,FRANCISCO FEBRES CORDERO,2016-2017,CUENCA,01H02002,FRANCISCO FEBRES CORDERO,2017-2018,1.00,NaN,1370
7,01H00144,ESPANA,2009-2010,CUENCA,01H00156,ESPANA,2010-2011,1.00,655.0,751
8,01H00151,JUAN MONTALVO,2011-2012,CUENCA,01H00155,JUAN MONTALVO,2012-2013,1.00,447.0,1788
9,01H00202,INEBHYE,2010-2011,CUENCA,01H01812,INEBHYE,2011-2012,1.00,NaN,97


### 7.1 Fusion confirmada vs. casos ambiguos

Un colegio solo puede fusionarse con **uno**. Si un AMIE desaparecido tiene
mas de un candidato que pasa todos los filtros (ej. varios colegios "VICENTE
ROCAFUERTE" en el mismo canton), no podemos saber a cual se fusiono
realmente -- ese caso se trata como **ambiguo** y se cuenta como cierre, no
como fusion confirmada.


In [9]:
conteo_candidatos = candidatos_df.groupby("amie_desaparecido").size() if len(candidatos_df) else pd.Series(dtype=int)

amies_fusion_confirmada = conteo_candidatos[conteo_candidatos == 1].index
amies_ambiguos = conteo_candidatos[conteo_candidatos > 1].index

fusion_confirmada = candidatos_df[candidatos_df["amie_desaparecido"].isin(amies_fusion_confirmada)].reset_index(drop=True)
candidatos_ambiguos = candidatos_df[candidatos_df["amie_desaparecido"].isin(amies_ambiguos)].sort_values("amie_desaparecido")

print(f"AMIE con UN solo candidato (fusion confirmada): {len(amies_fusion_confirmada):,}")
print(f"AMIE con MAS DE UN candidato (ambiguo -- se trata como cierre): {len(amies_ambiguos):,}")
print()
print("--- Ejemplo: 01B00019 (ROUSSEAU) ---")
display(fusion_confirmada[fusion_confirmada["amie_desaparecido"] == "01B00019"])
print()
print("--- Ejemplo de caso ambiguo (Vicente Rocafuerte, Guayaquil, si aplica) ---")
candidatos_ambiguos[candidatos_ambiguos["canton"] == "GUAYAQUIL"].head(10)


AMIE con UN solo candidato (fusion confirmada): 760
AMIE con MAS DE UN candidato (ambiguo -- se trata como cierre): 111

--- Ejemplo: 01B00019 (ROUSSEAU) ---


,amie_desaparecido,nombre_desaparecido,ultimo_periodo_visto,canton,amie_candidato,nombre_candidato,periodo_candidato,score_similitud,matricula_candidato_antes,matricula_candidato_despues
0,01B00019,ROUSSEAU,2009-2010,CUENCA,01H01665,ROUSSEAU,2010-2011,1.0,NaN,46



--- Ejemplo de caso ambiguo (Vicente Rocafuerte, Guayaquil, si aplica) ---


,amie_desaparecido,nombre_desaparecido,ultimo_periodo_visto,canton,amie_candidato,nombre_candidato,periodo_candidato,score_similitud,matricula_candidato_antes,matricula_candidato_despues
534,09H00262,PROVINCIA DEL GUAYAS,2013-2014,GUAYAQUIL,09H00581,PROVINCIA DEL GUAYAS,2014-2015,1.0,127.0,197
535,09H00262,PROVINCIA DEL GUAYAS,2013-2014,GUAYAQUIL,09H06158,PROVINCIA DEL GUAYAS,2014-2015,1.0,126.0,206
536,09H00267,JUAN XXIII,2015-2016,GUAYAQUIL,09H00353,JUAN XXIII,2016-2017,1.0,838.0,911
537,09H00267,JUAN XXIII,2015-2016,GUAYAQUIL,09H01424,JUAN XXIII,2016-2017,1.0,243.0,253
541,09H00355,MANUELA CANIZARES,2009-2010,GUAYAQUIL,09H02715,MANUELA CANIZARES,2010-2011,1.0,51.0,60
540,09H00355,MANUELA CANIZARES,2009-2010,GUAYAQUIL,09H01452,MANUELA CANIZARES,2010-2011,1.0,204.0,243
557,09H00704,REPUBLICA DE FRANCIA,2012-2013,GUAYAQUIL,09H01669,REPUBLICA DE FRANCIA,2013-2014,1.0,205.0,227
556,09H00704,REPUBLICA DE FRANCIA,2012-2013,GUAYAQUIL,09H00706,REPUBLICA DE FRANCIA,2013-2014,1.0,416.0,1801
563,09H00819,AMERICO VESPUCIO,2009-2010,GUAYAQUIL,09H00963,AMERICO VESPUCIO,2010-2011,1.0,81.0,118
564,09H00819,AMERICO VESPUCIO,2009-2010,GUAYAQUIL,09H02570,AMERICO VESPUCIO,2010-2011,1.0,168.0,369


## 7.2 Colegios cerrados (sin fusion confirmada)

Se consideran "cierre real" (no fusion) los posibles cierres que:
- No tienen ningun candidato que pase todos los filtros de la seccion 7, **o**
- Tienen mas de un candidato (ambiguo -- no se puede determinar a cual se
  fusiono, ver seccion 7.1).


In [10]:
amies_con_fusion_confirmada = set(fusion_confirmada["amie_desaparecido"])

cierres_probable_fusion = posibles_cierres[posibles_cierres["AMIE"].isin(amies_con_fusion_confirmada)]
cierres_reales = posibles_cierres[~posibles_cierres["AMIE"].isin(amies_con_fusion_confirmada)].sort_values(
    "ultimo_anio", ascending=False
)

print(f"Total posibles cierres: {len(posibles_cierres):,}")
print(f"  - Fusion confirmada (1 solo candidato, nombre casi identico y matricula creciente): {len(cierres_probable_fusion):,}")
print(f"  - Cierre real (sin candidato o candidato ambiguo, {len(amies_ambiguos):,} casos ambiguos incluidos aqui): {len(cierres_reales):,}")
print()
print("--- Colegios cerrados mas recientes (sin fusion confirmada) ---")
cierres_reales[["AMIE", "ultimo_nombre", "ultima_provincia", "ultimo_canton", "ultima_parroquia", "ultimo_periodo"]].head(20)


Total posibles cierres: 16,931
  - Fusion confirmada (1 solo candidato, nombre casi identico y matricula creciente): 760
  - Cierre real (sin candidato o candidato ambiguo, 111 casos ambiguos incluidos aqui): 16,171

--- Colegios cerrados mas recientes (sin fusion confirmada) ---


,AMIE,ultimo_nombre,ultima_provincia,ultimo_canton,ultima_parroquia,ultimo_periodo
27562,17H02072,ALONSO DE MERCADILLO,PICHINCHA,QUITO,SAN JOSE DE MINAS,2024-2025
29581,18H00144,LEON BECERRA,TUNGURAHUA,AMBATO,HUACHI LORETO,2024-2025
22218,13H03700,FISCAL 8 DE SEPTIEMBRE,MANABI,24 DE MAYO,SUCRE,2024-2025
20367,13H01474,JULIA ELOISA VERA NAVARRETE,MANABI,EL CARMEN,EL CARMEN,2024-2025
2097,02H00674,AURORA ESTRADA DE AYALA,BOLIVAR,LAS NAVES,LAS NAVES,2024-2025
2084,02H00660,PCEI PRIMERO DE MAYO,BOLIVAR,CALUMA,CALUMA,2024-2025
20092,13H01171,6 DE DICIEMBRE,MANABI,CHONE,CANUTO,2024-2025
29086,17H04379,CENTRO DE DESARROLLO INFANTIL LA CIGUENA,PICHINCHA,RUMIÑAHUI,SAN RAFAEL,2024-2025
19357,13H00279,HORIZONTES,MANABI,PORTOVIEJO,18 DE OCTUBRE,2024-2025
2196,02H00794,PCEI 23 DE AGOSTO,BOLIVAR,GUARANDA,ANGEL POLIBIO CHAVES,2024-2025


## 8. Validacion: caso Manuel Andrade Ureta

En 2009-2010 existian 3 AMIE: `13H00434` y `13H00447` (mismo nombre, mismo
canton/parroquia — Portoviejo/Andres de Vera) y `13H01397` (colegio distinto,
mismo nombre, en Chone/Convento). Desde 2010-2011 `13H00447` desaparece. El
algoritmo deberia sugerir `13H00434` como candidato — y **no** `13H01397`,
aunque comparta nombre, por estar en un canton distinto.


In [11]:
print("--- Historia de los 3 AMIE ---")
display(historia[historia["AMIE"].isin(["13H00434", "13H00447", "13H01397"])])

print()
print("--- Candidatos de fusion sugeridos para 13H00447 (antes de filtro de ambiguedad) ---")
display(candidatos_df[candidatos_df["amie_desaparecido"] == "13H00447"])

print()
print("--- Aparece en fusion_confirmada? ---")
display(fusion_confirmada[fusion_confirmada["amie_desaparecido"] == "13H00447"])


--- Historia de los 3 AMIE ---


,AMIE,primer_anio,ultimo_anio,primer_periodo,ultimo_periodo,nombres,cantones,parroquias,ultimo_nombre,ultimo_canton,ultima_parroquia,ultima_provincia,cambio_nombre,cambio_ubicacion,posible_inactivo
19468,13H00434,2009,2025,2009-2010,2025-2026,[MANUEL ANDRADE URETA],[PORTOVIEJO],"[ANDRÉS DE VERA, SAN PABLO]",MANUEL ANDRADE URETA,PORTOVIEJO,ANDRÉS DE VERA,MANABI,False,True,False
19475,13H00447,2009,2009,2009-2010,2009-2010,[MANUEL ANDRADE URETA],[PORTOVIEJO],[ANDRÉS DE VERA],MANUEL ANDRADE URETA,PORTOVIEJO,ANDRÉS DE VERA,MANABI,False,False,True
20299,13H01397,2009,2025,2009-2010,2025-2026,[MANUEL ANDRADE URETA],[CHONE],[CONVENTO],MANUEL ANDRADE URETA,CHONE,CONVENTO,MANABI,False,False,False



--- Candidatos de fusion sugeridos para 13H00447 (antes de filtro de ambiguedad) ---


,amie_desaparecido,nombre_desaparecido,ultimo_periodo_visto,canton,amie_candidato,nombre_candidato,periodo_candidato,score_similitud,matricula_candidato_antes,matricula_candidato_despues
854,13H00447,MANUEL ANDRADE URETA,2009-2010,PORTOVIEJO,13H00434,MANUEL ANDRADE URETA,2010-2011,1.0,47.0,180



--- Aparece en fusion_confirmada? ---


,amie_desaparecido,nombre_desaparecido,ultimo_periodo_visto,canton,amie_candidato,nombre_candidato,periodo_candidato,score_similitud,matricula_candidato_antes,matricula_candidato_despues
436,13H00447,MANUEL ANDRADE URETA,2009-2010,PORTOVIEJO,13H00434,MANUEL ANDRADE URETA,2010-2011,1.0,47.0,180


## 9. Matricula antes del cierre: ¿baja demanda o reubicacion?

Para cada posible cierre, se toma la matricula (`Total_Estudiantes`) de sus
hasta 3 ultimos anios registrados y se clasifica:
- **declive_baja_demanda**: la matricula del ultimo anio es baja (umbral
  ajustable, ver distribucion abajo) y venia en descenso — patron tipico de
  escuela unidocente rural que cierra por falta de estudiantes.
- **estable_o_alta**: la matricula del ultimo anio no es baja — el colegio no
  murio por falta de demanda, lo que apunta mas a fusion/reubicacion
  administrativa que a un cierre real.
- **sin_dato**: no hay matricula registrada para ese AMIE.


In [12]:
# Distribucion general de matricula, para justificar el umbral de "baja".
print("Percentiles de Total_Estudiantes (todo el panel):")
print(panel["total_estudiantes"].describe(percentiles=[0.1, 0.25, 0.5, 0.75]))

UMBRAL_MATRICULA_BAJA = 15  # escala tipica de escuela unidocente rural en Ecuador

panel_ordenado = panel.sort_values(["AMIE", "anio_inicio"])


def ultimos_valores(serie, n=3):
    return list(serie.tail(n))


matricula_historica = (
    panel_ordenado.groupby("AMIE")
    .agg(
        anios_matricula=("anio_inicio", lambda s: ultimos_valores(s, 3)),
        matriculas=("total_estudiantes", lambda s: ultimos_valores(s, 3)),
    )
    .reset_index()
)

posibles_cierres = posibles_cierres.merge(matricula_historica, on="AMIE", how="left")


def clasificar_tendencia(matriculas):
    valores = [m for m in matriculas if pd.notna(m)]
    if not valores:
        return "sin_dato"
    ultimo = valores[-1]
    no_creciente = len(valores) < 2 or all(
        valores[i] >= valores[i + 1] for i in range(len(valores) - 1)
    )
    if ultimo <= UMBRAL_MATRICULA_BAJA and no_creciente:
        return "declive_baja_demanda"
    if ultimo <= UMBRAL_MATRICULA_BAJA:
        return "baja_sin_tendencia_clara"
    return "estable_o_alta"


posibles_cierres["matricula_ultimo_anio"] = posibles_cierres["matriculas"].map(
    lambda vs: next((v for v in reversed(vs) if pd.notna(v)), None)
)
posibles_cierres["tendencia_matricula"] = posibles_cierres["matriculas"].map(clasificar_tendencia)

print()
print(f"Umbral de matricula baja usado: <= {UMBRAL_MATRICULA_BAJA} estudiantes")
print(posibles_cierres["tendencia_matricula"].value_counts())
print()
print("--- Ejemplos: declive_baja_demanda ---")
posibles_cierres[posibles_cierres["tendencia_matricula"] == "declive_baja_demanda"][
    ["AMIE", "ultimo_nombre", "ultimo_canton", "anios_matricula", "matriculas", "ultimo_anio"]
].head(10)


Percentiles de Total_Estudiantes (todo el panel):
count    341363.000000
mean        218.915064
std         413.275554
min           0.000000
10%          11.000000
25%          21.000000
50%          57.000000
75%         223.000000
max        9462.000000
Name: total_estudiantes, dtype: float64

Umbral de matricula baja usado: <= 15 estudiantes
tendencia_matricula
estable_o_alta              11558
declive_baja_demanda         4172
baja_sin_tendencia_clara     1201
Name: count, dtype: int64

--- Ejemplos: declive_baja_demanda ---


,AMIE,ultimo_nombre,ultimo_canton,anios_matricula,matriculas,ultimo_anio
0,01B00007,HUELLITAS CREATIVAS,CUENCA,[2010],[1],2010
3,01B00020,NARANCAY,CUENCA,"[2018, 2019, 2020]","[14, 13, 5]",2020
6,01B00028,ESTRELLA DE SAN JOSE,CUENCA,"[2014, 2015, 2016]","[11, 9, 9]",2016
14,01B00059,OSWALDO GUAYASAMIN,CAMILO PONCE ENRIQUEZ,"[2019, 2020, 2021]","[5, 5, 5]",2021
16,01B00064,FROILAN ABAD AMBROSI,CUENCA,[2009],[7],2009
17,01B00068,None,NABON,[2009],[7],2009
25,01H00027,ACORDES,CUENCA,[2020],[0],2020
31,01H00039,UNIVERSO DE COLORES,CUENCA,[2009],[8],2009
35,01H00045,DE BACHILLERATO PARTICULAR JOSE MARIA VELAZ PCEI,CUENCA,"[2016, 2017, 2018]","[88, 46, 7]",2018
37,01H00051,INTEGRAL,CUENCA,[2009],[15],2009


## 10. Resumen y limitaciones

- Se cargaron **17 archivos** (MINEDUC 2009-2010 a 2023-2024, MINEDEC 2024-2025
  y 2025-2026 Inicio), seleccionando solo columnas de identidad/ubicacion y
  matricula por **posicion** (no por nombre de columna, ya que los nombres
  varian levemente entre anios en tildes/espacios/orden de columnas extra).
- La normalizacion de nombres usa una lista de prefijos institucionales
  comunes; es heuristica y puede necesitar ajustes si aparecen mas casos de
  ruido (avisar si se detectan patrones nuevos).
- Se observaron **cambios de un solo anio que luego revierten** (ej. una
  parroquia que cambia y vuelve al valor anterior al anio siguiente) — posible
  error de captura del Ministerio en ese periodo, no necesariamente un cambio
  real. Revisar la tabla de transiciones (seccion 6) caso por caso.
- **"Fusion" se define unicamente por nombre similar/identico en el mismo
  canton (seccion 7, ej. Manuel Andrade Ureta)** — son sugerencias para
  revision manual, no conclusiones automaticas.
- Se probo tambien un metodo de consolidacion muchos-a-uno basado solo en que
  la matricula de un codigo nuevo coincidiera con la suma de varios codigos
  cerrados en la misma parroquia (patron tipico de "Unidad Educativa del
  Milenio"). **Se descarto como criterio de fusion**: sin una senal de nombre,
  no se puede saber con certeza cuales colegios especificos fueron absorbidos
  por el codigo nuevo — la coincidencia numerica sola no es prueba suficiente.
- El umbral de "matricula baja" (seccion 9, <=15 estudiantes) es ajustable —
  se eligio con base en la escala tipica de escuelas unidocentes rurales, no
  en un calculo estadistico exhaustivo.
- **Pendiente:** integrar `matricula-pension-mayo-2026.xlsx`. Ese archivo solo
  cubre colegios con pension (particulares), a diferencia de los censos
  completos — se debe decidir su tratamiento sin generar falsos "cierres" de
  colegios fiscales/municipales.


## 11. Exportar resultados a Excel

Tabla final, una fila por AMIE, con un estado resumen:
- **Activo**: sigue apareciendo en el ultimo periodo (2025-2026).
- **Fusionado**: desaparecio y tiene una **fusion confirmada** (seccion 7.1 --
  nombre casi identico, no generico, matricula creciente, un unico candidato).
  Se incluye el AMIE y nombre del colegio al que se fusiono.
- **Cerrado**: desaparecio sin fusion confirmada -- sin candidato, o con
  candidatos ambiguos (se listan en `candidatos_ambiguos` para revision manual,
  por si acaso, pero no se asigna ninguno como definitivo).


In [13]:
amie_a_fusion = fusion_confirmada.set_index("amie_desaparecido")[["amie_candidato", "nombre_candidato"]]

ambiguos_por_amie = (
    candidatos_ambiguos.groupby("amie_desaparecido")
    .apply(lambda g: "; ".join(f"{a} ({n})" for a, n in zip(g["amie_candidato"], g["nombre_candidato"])))
    if len(candidatos_ambiguos)
    else pd.Series(dtype=str)
)

exportar = historia.copy()

exportar["estado"] = "Activo"
exportar.loc[exportar["posible_inactivo"], "estado"] = "Cerrado"
exportar.loc[exportar["AMIE"].isin(amie_a_fusion.index), "estado"] = "Fusionado"

exportar["amie_fusion"] = exportar["AMIE"].map(amie_a_fusion["amie_candidato"])
exportar["nombre_fusion"] = exportar["AMIE"].map(amie_a_fusion["nombre_candidato"])
exportar["candidatos_ambiguos"] = exportar["AMIE"].map(ambiguos_por_amie)

columnas_export = [
    "AMIE", "ultimo_nombre", "ultima_provincia", "ultimo_canton", "ultima_parroquia",
    "primer_periodo", "ultimo_periodo", "estado", "amie_fusion", "nombre_fusion",
    "candidatos_ambiguos", "cambio_nombre", "cambio_ubicacion",
]
exportar_final = exportar[columnas_export].sort_values(
    ["estado", "ultimo_periodo"], ascending=[True, False]
).reset_index(drop=True)

print(exportar_final["estado"].value_counts())

RUTA_SALIDA = PROJECT_ROOT / "data" / "processed" / "historial_colegios_mineduc.xlsx"
exportar_final.to_excel(RUTA_SALIDA, index=False, sheet_name="Historial AMIE")
print(f"Guardado en: {RUTA_SALIDA}")
exportar_final.head(10)


estado
Activo       16275
Cerrado      16171
Fusionado      760
Name: count, dtype: int64


C:\Users\vanessa.gallegos\AppData\Local\Temp\ipykernel_4108\2571145429.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: "; ".join(f"{a} ({n})" for a, n in zip(g["amie_candidato"], g["nombre_candidato"])))


Guardado en: c:\Users\vanessa.gallegos\2026\proyecto_colegios\data\processed\historial_colegios_mineduc.xlsx


,AMIE,ultimo_nombre,ultima_provincia,ultimo_canton,ultima_parroquia,primer_periodo,ultimo_periodo,estado,amie_fusion,nombre_fusion,candidatos_ambiguos,cambio_nombre,cambio_ubicacion
0,01B00021,EDUCACION BASICA SEIS DE JUNIO,AZUAY,CUENCA,CHAUCHA,2009-2010,2025-2026,Activo,NaN,NaN,NaN,True,False
1,01B00027,MONSENOR LEONIDAS PROANO II,AZUAY,CUENCA,QUINGEO,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,False
2,01B00029,EL TEJAR,AZUAY,CUENCA,QUINGEO,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,False
3,01B00030,BUENA ESPERANZA,AZUAY,CUENCA,RICAURTE,2009-2010,2025-2026,Activo,NaN,NaN,NaN,True,False
4,01B00031,MONTE SINAI,AZUAY,CUENCA,SANTA ANA,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,False
5,01B00032,XAVIER MUNOZ CHAVEZ,AZUAY,CUENCA,TARQUI,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,True
6,01B00034,CESAR VICTORIANO MARTINEZ VERDUGO,AZUAY,CUENCA,TURI,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,False
7,01B00035,MARCO MARTINEZ,AZUAY,CUENCA,TURI,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,False
8,01B00036,JUAN PENAFIEL,AZUAY,CUENCA,VALLE,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,True
9,01B00037,MIGUEL IGNACIO CORDERO MONCAYO,AZUAY,GUALACEO,GUALACEO,2009-2010,2025-2026,Activo,NaN,NaN,NaN,False,False
